# Part 10: Testing the final model on our own phone recordings

> **Status: skeleton.** Waiting for (1) the final model choice from Part 7 and (2) our phone recordings (see `09_phone_pipeline.ipynb`, Step 4). Every cell runs now; the cells that need missing inputs print what they are waiting for instead of failing.

**Question:** does a model trained on the 21 UCI training people still work on **different people, different phones, recorded by us**?

**Data used:**
- **Training:** the 21 UCI training people only, as features recomputed by our pipeline (Part 9).
- **Testing:** our own phone recordings in `data/phone/<person>/`.
- The 9 UCI test people (`test/`) are **not** used here. They belong to Part 7's single final test.

## Step 1: load the pipeline from Part 9
`%run` executes `09_phone_pipeline.ipynb` and gives us its functions (`load_person`, `compute_features`, `to_uci_scale`, `check_orientation`) and its training-set results (`scaled`, `y`, `good`, `feature_names`).

⚠️ Takes about 2 minutes (Part 9 recomputes the features of all 7,352 training windows).

In [1]:
from IPython.utils.capture import capture_output

# Part 9's printed output and plots are captured so this notebook stays short;
# run part9_output.show() in a new cell to see them.
with capture_output() as part9_output:
    %run ./09_phone_pipeline.ipynb

In [2]:
print("training windows:", scaled.shape[0], "| features the phone can deliver (reproduced):", good.sum())

training windows: 7352 | features the phone can deliver (reproduced): 451


## Step 2: load the phone recordings
One folder per person in `data/phone/`. Each person's windows get the same processing as in Part 9. We also check the orientation of the phone against UCI's standing windows (Part 9, Check 5).

In [3]:
PHONE_DIR = "data/phone"
AXIS_OPTIONS = {}   # TODO after the orientation check, e.g. {"lakshika": dict(axis_map=(1, 0, 2), axis_sign=(1, -1, 1))}

phone_X, phone_y, phone_person = [], [], []
people = sorted(d for d in os.listdir(PHONE_DIR) if os.path.isdir(os.path.join(PHONE_DIR, d))) if os.path.isdir(PHONE_DIR) else []

for person in people:
    print(person)
    features, labels = load_person(os.path.join(PHONE_DIR, person), **AXIS_OPTIONS.get(person, {}))
    check_orientation(features, labels)
    phone_X.append(features)
    phone_y.append(labels)
    phone_person += [person] * len(labels)

if people:
    phone_X, phone_y, phone_person = np.vstack(phone_X), np.concatenate(phone_y), np.array(phone_person)
    print("phone windows:", len(phone_y), "| per activity:", {activity[k]: int((phone_y == k).sum()) for k in np.unique(phone_y)})
else:
    print("WAITING: no recordings found in", PHONE_DIR, "(see 09_phone_pipeline.ipynb, Step 4)")

WAITING: no recordings found in data/phone (see 09_phone_pipeline.ipynb, Step 4)


## Step 3: the final model (from Part 7)
The phone can only deliver the **451 reproduced features** (Part 9), so we **retrain** the model type that Part 7 picks, with its chosen setting, on those features of the 21 training people. Part 9 showed this costs almost nothing for logistic regression (0.931 vs 0.933).

TODO: replace `None` with Part 7's choice. The candidates so far (one-SE choices from Parts 4–6):
- logistic lasso: `make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=0.1, max_iter=5000))`
- RBF SVM: `make_pipeline(StandardScaler(), SVC(C=1, gamma=0.001))`
- random forest: `RandomForestClassifier(n_estimators=100, max_depth=10, max_features="sqrt", random_state=SEED, n_jobs=-1)`

In [4]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

final_model = None   # TODO (Part 7): put the chosen model here
model_name = "TODO"  # TODO (Part 7): e.g. "svm_rbf C=1 gamma=0.001"

if final_model is None:
    print("WAITING: final model from Part 7")
else:
    final_model.fit(scaled[:, good], y)
    print("trained", model_name, "on", len(y), "training windows,", good.sum(), "features")

WAITING: final model from Part 7


## Step 4: predict on the phone windows
Overall macro-F1, a confusion matrix (rows = true activity, columns = predicted), and macro-F1 per person. For comparison, Part 7's cross-validation and final-test scores are on UCI's people and phone.

In [5]:
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

ready = final_model is not None and len(people) > 0
if not ready:
    print("WAITING:", "final model (Part 7)" if final_model is None else "", "| recordings" if not people else "")
else:
    phone_pred = final_model.predict(phone_X[:, good])
    print("phone macro-F1:", round(f1_score(phone_y, phone_pred, average="macro"), 3),
          "| accuracy:", round((phone_pred == phone_y).mean(), 3))
    for person in people:
        mine = phone_person == person
        print(f"  {person:10s} macro-F1 {f1_score(phone_y[mine], phone_pred[mine], average='macro'):.3f}  ({mine.sum()} windows)")

    labels = sorted(activity)
    ConfusionMatrixDisplay(confusion_matrix(phone_y, phone_pred, labels=labels),
                           display_labels=[activity[k] for k in labels]).plot(cmap="Blues", xticks_rotation=30)
    plt.title("Phone recordings: " + model_name)
    plt.show()

WAITING: final model (Part 7) | recordings


## TODO list
- [ ] **Recordings:** record all 6 activities (instructions in Part 9, Step 4) and put them in `data/phone/<person>/`. They are committed with the repo so others can rerun this notebook (keep them as plain CSV files).
- [ ] **Orientation:** run Step 2, compare the standing gravity with UCI's; if it differs, set `AXIS_OPTIONS` and rerun.
- [ ] **Final model:** fill in `final_model` and `model_name` in Step 3 from Part 7.
- [ ] **Results:** run Restart & Run All; write the summary below (macro-F1 on our phones vs Part 7's numbers; which activities get confused; per person).
- [ ] **Discussion:** if scores drop, say why (different phone and sensor noise, different wearing position, different people, the 110 features we cannot reproduce, the noise-shape features of still activities flagged in Part 9).

## Summary
*(to be written after the run, with numbers from the cells above)*